# درس 11: فرایند پواسون و رسیدن سفارش‌ها

متن کامل درس: [https://quantacademy.ir/courses/quant-math/poisson-process-order-arrivals/](https://quantacademy.ir/courses/quant-math/poisson-process-order-arrivals/)

داده‌های این دوره **شبیه‌سازی‌شده** هستند، نه قیمت واقعی بازار؛ هدف یاد دادن روش است. آموزشی است و توصیه سرمایه‌گذاری نیست.

سلول‌ها را از بالا به پایین اجرا کن (Shift+Enter).

## ۱. فرایند پواسون

## ۲. شبیه‌سازی

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(11)
lam, T = 0.2, 3600.0
gaps = rng.exponential(1 / lam, 2 * int(lam * T))
times = np.cumsum(gaps)
times = times[times < T]
counts = np.bincount((times // 60).astype(int), minlength=60)
print(f"events {len(times)}   expected {lam * T:.0f}")
print(f"per-minute counts: mean {counts.mean():.2f}  variance {counts.var(ddof=1):.2f}  (Poisson: both {lam * 60:.1f})")
g = np.diff(times)
print(f"gaps: mean {g.mean():.2f} (1/lambda = {1 / lam:.2f})  std {g.std():.2f}  median {np.median(g):.2f} (ln2/lambda = {np.log(2) / lam:.2f})")
print(f"P(gap > 10s) {np.mean(g > 10):.3f}   exp(-lambda*10) = {np.exp(-lam * 10):.3f}")

## ۳. بی‌حافظگی

In [ ]:
s, t = 5.0, 5.0
survivors = g[g > s]
print(f"P(gap > {s + t:.0f} | gap > {s:.0f}) = {np.mean(survivors > s + t):.3f}   P(gap > {t:.0f}) = {np.mean(g > t):.3f}")

## ۴. نرخ رسیدن معامله‌ها در داده

In [ ]:
URL = "https://raw.githubusercontent.com/quantacademyir/quant-math/main/data/trades_sample.csv"
tr = pd.read_csv(URL, parse_dates=["timestamp"])
tr["sec"] = (tr["timestamp"] - tr["timestamp"].dt.normalize().iloc[0]).dt.total_seconds()
print(len(tr), tr["timestamp"].iloc[0], tr["timestamp"].iloc[-1])
per_min = tr.groupby((tr["sec"] // 60).astype(int)).size().reindex(range(1440), fill_value=0)
print(f"whole day: mean {per_min.mean():.1f} per minute, variance {per_min.var(ddof=1):.1f}")
for name, a, b in (("quiet 03-04", 3, 4), ("busy 14-15", 14, 15)):
    c = per_min.iloc[a * 60:b * 60]
    n = c.sum()
    print(f"{name}: trades {n:5d}  rate {n / 3600:.3f}/s +- {2 * np.sqrt(n) / 3600:.3f}   per-minute mean {c.mean():5.1f} variance {c.var(ddof=1):6.1f}  Fano {c.var(ddof=1) / c.mean():.2f}")

## ۵. فاصله‌ها دم سنگین‌تر از نمایی دارند

In [ ]:
busy = tr[(tr["sec"] >= 14 * 3600) & (tr["sec"] < 15 * 3600)]
gaps_b = np.diff(busy["sec"].values)
gaps_b = gaps_b[gaps_b > 0]
print(f"busy hour gaps: n {len(gaps_b)} mean {gaps_b.mean():.2f}s std {gaps_b.std():.2f}s  CV {gaps_b.std() / gaps_b.mean():.2f} (Poisson: 1.00)")
for x in (1, 5, 10, 20):
    print(f"P(gap > {x:>2d}s) empirical {np.mean(gaps_b > x):.3f}   exponential {np.exp(-x / gaps_b.mean()):.3f}")

## ۶. Thinning و Superposition

In [ ]:
share_buy = (tr["side"] == "buy").mean()
print(f"share of buys {share_buy:.3f}   buys {int((tr.side == 'buy').sum())}  sells {int((tr.side == 'sell').sum())}")
rng = np.random.default_rng(12)
total = rng.poisson(0.3 * 60, 20000)
buys = rng.binomial(total, 0.55)
sells = total - buys
print(f"thinning (simulated): buys mean {buys.mean():.2f} (0.55*18 = {0.55 * 18:.2f}) var {buys.var():.2f}   corr(buys, sells) = {np.corrcoef(buys, sells)[0, 1]:+.3f}")
bm = busy.groupby((busy["sec"] // 60).astype(int))["side"].apply(lambda s: (s == "buy").sum())
sm = busy.groupby((busy["sec"] // 60).astype(int))["side"].apply(lambda s: (s == "sell").sum())
print(f"busy hour data: corr(buys per minute, sells per minute) = {np.corrcoef(bm, sm)[0, 1]:+.3f}")

## ۷. احتمال پر شدن سفارش

In [ ]:
sells = busy[busy["side"] == "sell"]["sec"].values
lam_s = len(sells) / 3600
tau = 10
edges = np.arange(14 * 3600, 15 * 3600 + tau, tau)
hit = np.histogram(sells, bins=edges)[0] > 0
print(f"sell rate {lam_s:.3f}/s   P(fill in {tau}s): Poisson model {1 - np.exp(-lam_s * tau):.3f}   empirical {hit.mean():.3f}")